In [9]:
import numpy as np
import pandas as pd

In [10]:
#Loading data 
ACLED_df = pd.read_csv('../Spatial Joins/ACLED Join/joined_ACLED_ZIP_22_PA.csv')

#### Restructuring Date Column

In [11]:
#Extracting year and month from date column

#Parsing date column to datetime
ACLED_df['parsed_date'] = pd.to_datetime(ACLED_df['event_date'], format='%d %B %Y')

#Extracting year and month
ACLED_df['event_MY'] = ACLED_df['parsed_date'].dt.to_period('M')

#### Aggregation function 
Can add more if needed

In [12]:
#Defining aggregation across features
aggregation_function = {
    'admin2': 'first', 
    'fatalities': 'sum'  
}

#### Grouping by ZIP Code and Month

In [13]:
#Grouping by ZIP Code and Month
#agg aggregates over the previously specified parameters
grouped = ACLED_df.groupby(['ZCTA5CE10', 'event_MY']).agg(aggregation_function).reset_index()

#Adding event count per ZIP Code and Month
grouped['event_count'] = ACLED_df.groupby(['ZCTA5CE10', 'event_MY']).size().reset_index(name='event_count')['event_count']

print(grouped.head(10))

   ZCTA5CE10 event_MY        admin2  fatalities  event_count
0      15001  2022-01        Beaver           0            1
1      15001  2022-09        Beaver           0            1
2      15009  2022-06        Beaver           0            2
3      15010  2022-05        Beaver           0            1
4      15025  2022-05     Allegheny           0            1
5      15026  2022-10     Allegheny           0            1
6      15026  2022-12     Allegheny           0            1
7      15037  2022-07     Allegheny           0            1
8      15068  2022-03  Westmoreland           0            1
9      15068  2022-07  Westmoreland           0            3


In [23]:
#Creating an index 'date range' of all months (in order to allocate each ZIP code to each month) 
date_range = pd.period_range(start='2022-01', end='2022-12', freq='M')
print(date_range)

PeriodIndex(['2022-01', '2022-02', '2022-03', '2022-04', '2022-05', '2022-06',
             '2022-07', '2022-08', '2022-09', '2022-10', '2022-11', '2022-12'],
            dtype='period[M]')


In [15]:
#Dataframe mapping ZIP Codes to Counties 
zip_county = ACLED_df[['ZCTA5CE10', 'admin2']].drop_duplicates()

In [16]:
#New dataframe combining date range and ZIP Code
#Using multiindex object to combine unique ZIP code list with date range index 
all_combinations = pd.MultiIndex.from_product([ACLED_df['ZCTA5CE10'].unique(), date_range], names=['ZCTA5CE10', 'event_MY'])

In [24]:
#Initializing df with all combinations 
#Resetting index for default index 
restructured_df = pd.DataFrame(index=all_combinations).reset_index()

In [26]:
#Merge 1
#Merging zip_county df with initialized df for all combinations
restructured_df = restructured_df.merge(zip_county, on='ZCTA5CE10', how='left')

In [27]:
#Merge 2 
#Merging initialized df with grouped data holding aggregates and counts
restructured_df = restructured_df.merge(grouped, on=['ZCTA5CE10', 'admin2', 'event_MY'], how='left')

In [28]:
#Filling missing values + converting to int
restructured_df['event_count'] = restructured_df['event_count'].fillna(0).astype(int)
restructured_df['fatalities'] = restructured_df['fatalities'].fillna(0).astype(int)

In [29]:
#Sorting df by ZIP and time
restructured_df = restructured_df.sort_values(by=['ZCTA5CE10', 'event_MY'])
print(restructured_df.head(10))

     ZCTA5CE10 event_MY  admin2  fatalities  event_count
564      15001  2022-01  Beaver           0            1
565      15001  2022-02  Beaver           0            0
566      15001  2022-03  Beaver           0            0
567      15001  2022-04  Beaver           0            0
568      15001  2022-05  Beaver           0            0
569      15001  2022-06  Beaver           0            0
570      15001  2022-07  Beaver           0            0
571      15001  2022-08  Beaver           0            0
572      15001  2022-09  Beaver           0            1
573      15001  2022-10  Beaver           0            0
